# CSE 25 Discussion 4 Notebook
## Probability Fundamentals for Language Models

This is an **optional practice** notebook.

Goal: strengthen your probability intuition with short coding exercises, then connect that to language-model probability.

## How this discussion is structured
- **Part A**: recap + practice on events, conditional probability, independence, and sequence probability (with and without replacement).
- **Part B**: connect probability ideas to language models (word distributions, next-token conditional probability, and chain-rule sentence probability).
- We will use Python + NumPy (mostly), with easy toy examples in Pandas and scikit-learn.

Math conventions used throughout:
- Sample space: $\Omega$: the full set of outcomes that could happen in one trial.
- Event probability: $P(A)$: how likely event $A$ is, as a number between 0 and 1.
- Conditional probability: $P(A\mid B)=\frac{P(A,B)}{P(B)}$ (when $P(B)>0$): once we know $B$ happened, what fraction of those cases also satisfy $A$?
- Complement rule: $P(\neg A)=1-P(A)$: probability that $A$ does not happen.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

## Part A: Recap + Practice

Core formulas for this part:
- Uniform counting idea: $P(A)=\frac{|A|}{|\Omega|}$: count outcomes in event $A$, then divide by total outcomes.
- Disjoint add rule: $P(A\cup B)=P(A)+P(B)$ if $A\cap B=\varnothing$: if two events cannot happen together, probability of "A or B" is the sum.
- Product rule: $P(A,B)=P(A\mid B)P(B)$: probability of both events equals "chance of $B$" times "chance of $A$ given $B$."
- Independence check: $A\perp B \iff P(A\mid B)=P(A) \iff P(A,B)=P(A)P(B)$: if learning $B$ gives no new information about $A$, they are independent.

In [ ]:
# Toy box of shapes (same style as lecture examples)
counts = {
    'blue_square': 2, 'blue_triangle': 2, 'blue_circle': 2,
    'red_square': 3,  'red_triangle': 3,  'red_circle': 3,
    'yellow_square': 3, 'yellow_triangle': 3, 'yellow_circle': 3,
}
total = sum(counts.values())
print('Total objects:', total)

### Exercise 1: Events (short recap)
Fill in the blanks to compute event probabilities.

Use:
- $P(blue)=\frac{\#\text{ blue objects}}{\#\text{ total objects}}$
- $P(square\ \text{or}\ triangle)=P(square)+P(triangle)$ (they are disjoint by shape)

- $P(A)=\frac{|A|}{|\Omega|}$: event probability is "favorable outcomes over all outcomes."
- $P(square\cup triangle)=P(square)+P(triangle)$: a shape cannot be both square and triangle at once, so the add rule applies directly.

In [ ]:
blue_count = ...
square_count = ...
triangle_count = ...

p_blue = ...
p_square_or_triangle = ...

assert blue_count == 6, 'blue_count should be 6'
assert np.isclose(p_blue, 6 / 24), 'P(blue) should be 6/24'
assert np.isclose(p_square_or_triangle, 16 / 24), 'P(square or triangle) should be 16/24'

### Exercise 2: Conditional probability
Compute \(P(blue\mid square)\) using \(P(A\mid B)=P(A,B)/P(B)\).

In this exercise:
- $A=\{blue\}$
- $B=\{square\}$
- So $P(blue\mid square)=\frac{P(blue\cap square)}{P(square)}$

- $P(blue\mid square)$: first pretend you are only looking at squares; inside that reduced world, what fraction are blue?

In [ ]:
p_blue_and_square = ...
p_square = ...
p_blue_given_square = ...

assert np.isclose(p_blue_and_square, 2 / 24), 'P(blue and square) should be 2/24'
assert np.isclose(p_blue_given_square, 1 / 4), 'P(blue|square) should be 1/4'

### Exercise 3: Independence check
Events \(A=blue\) and \(B=square\) are independent if \(P(A\mid B)=P(A)\).

Equivalent check:
- $P(blue\cap square)=P(blue)P(square)$

- $P(A\mid B)=P(A)$: if events are independent, knowing one happened does not change the probability of the other.

In [ ]:
p_blue = 6 / 24
p_blue_given_square = 2 / 8

independent = ...

assert independent is True, 'blue and square are independent in this toy box'

### Exercise 4: Sequence probability with replacement
Compute \(P(red\_triangle, yellow\_circle, red\_triangle)\).

Because sampling is **with replacement**, each draw has the same distribution and draws are independent:
- $P(x_1,x_2,x_3)=P(x_1)P(x_2)P(x_3)$

- $P(x_1,x_2,x_3)=P(x_1)P(x_2)P(x_3)$: with replacement, you put the object back each time, so probabilities do not change from draw to draw.

In [ ]:
p_red_triangle = 3 / 24
p_yellow_circle = 3 / 24

seq_prob = ...

assert np.isclose(seq_prob, (3 / 24) * (3 / 24) * (3 / 24)), 'Use product rule with replacement'

### Exercise 4B: Sequence probability without replacement
Now compute the same sequence \(P(red\_triangle, yellow\_circle, red\_triangle)\) **without** replacement.

Without replacement, probabilities change after each draw because counts and total objects decrease:
- $P(x_1,x_2,x_3)=P(x_1)P(x_2\mid x_1)P(x_3\mid x_1,x_2)$: multiply conditional probabilities where each term reflects updated counts.

In [ ]:
# From the toy box: red_triangle=3, yellow_circle=3, total=24
# Sequence: red_triangle, yellow_circle, red_triangle (without replacement)
p1 = ...            # first red triangle
p2_given_p1 = ...   # then yellow circle, after removing one red triangle
p3_given_p1p2 = ... # then red triangle, after removing one red triangle and one yellow circle

seq_prob_no_replace = ...

assert np.isclose(p1, 3 / 24), 'First draw should be 3/24'
assert np.isclose(p2_given_p1, 3 / 23), 'Second draw should be 3/23'
assert np.isclose(p3_given_p1p2, 2 / 22), 'Third draw should be 2/22'
assert np.isclose(seq_prob_no_replace, (3 / 24) * (3 / 23) * (2 / 22)), 'Use changing denominators without replacement'

### Exercise 5: NumPy practice on probabilities
Use vectorized operations to compute normalized probabilities and event probability sums.

If a count vector is $c=[c_1,\dots,c_k]$, then probability vector is
$p=\frac{c}{\sum_i c_i}$ and must satisfy $\sum_i p_i=1$.

- $p_i=\frac{c_i}{\sum_j c_j}$: divide each count by the total count to convert frequencies into probabilities.

In [ ]:
# [blue, red, yellow] counts from the same box
color_counts = np.array([6, 9, 9])
probs = ...
p_not_red = ...

assert np.isclose(probs.sum(), 1.0), 'Probabilities must sum to 1'
assert np.isclose(probs[0], 6 / 24), 'Blue probability should be 6/24'
assert np.isclose(p_not_red, 18 / 24), 'P(not red) should be 18/24'

## Part B: Connecting probability concepts to language models

How the ideas map to language modeling:
- Discrete distribution over vocabulary: $P(w)=\frac{count(w)}{\sum_{v\in V}count(v)}$: a language model can treat words as outcomes and assign probabilities from counts.
- Conditional next-word probability: $P(w_t\mid h)$ where $h=(w_1,\dots,w_{t-1})$: probability changes based on history.
- Chain rule for sentence probability: $P(w_1,\dots,w_n)=\prod_{k=1}^n P(w_k\mid w_{1:k-1})$: multiply next-word probabilities to get full sentence probability.
- Ranking idea: if two candidate sentences come from the same model, the one with larger probability is the model's preferred sentence.

### Exercise 6: From word counts to a unigram language model
Use counts to build a probability distribution over words.

- $P(w)=\frac{count(w)}{\sum_{v\in V}count(v)}$: each word gets probability equal to its relative frequency.
- In this simplified unigram model, each next word is sampled from the same vocabulary distribution.

In [ ]:
# Toy corpus counts
vocab = np.array(['the', 'cat', 'sat', 'mat', '.'])
counts = np.array([8, 3, 2, 2, 1])

probs = ...
p_the = ...
p_cat = ...

assert np.isclose(probs.sum(), 1.0), 'Word probabilities must sum to 1'
assert np.isclose(p_the, 8 / 16), 'P(the) should be 8/16'
assert np.isclose(p_cat, 3 / 16), 'P(cat) should be 3/16'

### Exercise 7: Conditional probabilities and chain rule in a language model
Compute a short sentence probability using history-dependent next-word probabilities.

- $P(w_1,w_2,w_3)=P(w_1)P(w_2\mid w_1)P(w_3\mid w_1,w_2)$: chain rule multiplies step-by-step token probabilities.
- Interpretation: the next token probability depends on context, not just the token by itself.

In [ ]:
# Suppose a toy language model gives:
p_the = 0.5
p_cat_given_the = 0.6
p_dot_given_the_cat = 0.5

# Probability of sentence: "the cat ."
p_sentence = ...

assert np.isclose(p_sentence, 0.5 * 0.6 * 0.5), 'Apply chain rule to sentence probability'
assert np.isclose(p_sentence, 0.15), 'Expected numeric value is 0.15'

### Exercise 8: Class probabilities and next-token probabilities
Use scikit-learn probabilities to mirror language-model next-token distributions.

- $\sum_w P(w\mid h)=1$: for a fixed context/history, probabilities across all candidate next tokens must sum to 1.
- `predict_proba` in a classifier also returns a distribution over classes that sums to 1.

Connection: classes in classification behave like candidate next tokens in language modeling.

Here, we represent each context using the previous two words (`prev2`, `prev1`) so the task feels like sentence continuation.

In [ ]:
# Toy sentence contexts: (prev2, prev1) -> next_token
df = pd.DataFrame({
    'prev2': ['the', 'the', 'a', 'a', 'the', 'a'],
    'prev1': ['cat', 'dog', 'cat', 'dog', 'cat', 'dog'],
    'next_token': ['sat', 'barked', 'sat', 'barked', 'sat', 'barked'],
})

# 1) Pandas: empirical unigram probability P(next_token='sat')
p_sat = ...
assert np.isclose(p_sat, 3 / 6), 'P(next_token=sat) should be 3/6'

# 2) scikit-learn: one-hot encode context words and predict next-token distribution
X = pd.get_dummies(df[['prev2', 'prev1']])
y = df['next_token']

model = DecisionTreeClassifier(max_depth=1, random_state=0)
model.fit(X, y)

# Predict distribution for context: "the dog ..."
query = pd.DataFrame([{'prev2': 'the', 'prev1': 'dog'}])
query_X = pd.get_dummies(query).reindex(columns=X.columns, fill_value=0)
proba = model.predict_proba(query_X)[0]
predicted_token = model.classes_[np.argmax(proba)]

assert np.isclose(proba.sum(), 1.0), 'Distribution over next-token candidates must sum to 1'
assert predicted_token == 'barked', 'For context "the dog", the most likely next token should be barked'

## Wrap-up
- You practiced probability events, conditional probability, independence checks, and replacement sequences.
- You connected those ideas to language modeling: word distributions, next-token conditionals, and chain-rule sentence probability.
- You also used NumPy, Pandas, and scikit-learn to see how probability distributions appear in code.

If you finish early, re-run your cells top-to-bottom and explain each probability in plain language.